In [1]:
# Terminal
# pip install scikit-learn==1.1.3
# pip install missingpy==0.2.0
# Restart Kernel

In [2]:
pip install gower

  Using cached gower-0.1.2-py3-none-any.whl.metadata (3.7 kB)
Using cached gower-0.1.2-py3-none-any.whl (5.2 kB)

[notice] A new release of pip is available: 24.1.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

def onehot_to_ord_df(df: pd.DataFrame):
    """
    Convert one-hot categorical dummy columns into ordinal columns.
    
    Original data (OneHotEncoder(drop='first')):
    - categorical_0,...,categorical_3 are binary -> one dummy each: categorical_0_Yes,..., categorical_3_Yes (0/1) baseline "No"
    - categorical_4 is 3 level multinomial -> two dummies categorical_4_Level_B and categorical_4_Level_C baseline "Level_A" when B=0,C=0

    Converts into:
    - categorical_0,...categorical_3 become ordinal columns in {0,1} (NaN stays NaN)
    - categorical_4 becomes ordinal column in {0,1,2}  (A=0,B=1,C=2) (NaN stays NaN)
    """
    df = df.copy()

    # find dummy columns in dataset
    binary_dummy_cols = []
    for i in range(4):
        col_yes = f"categorical_{i}_Yes"
        if col_yes in df.columns:
            binary_dummy_cols.append((i, col_yes))

    col_b = "categorical_4_Level_B"
    col_c = "categorical_4_Level_C"
    multi_dummy_cols = [col_b, col_c] if (col_b in df.columns and col_c in df.columns) else []

    meta = {
        "all_dummy_cols": [c for _, c in binary_dummy_cols] + multi_dummy_cols,
        "original_columns": df.columns.tolist(),
    }

    # binary to ordinal: 0/1 (keep NaN if missing)
    for i, col_yes in binary_dummy_cols:
        new_col = f"categorical_{i}"
        y = df[col_yes].astype(float)
        df[new_col] = np.where(y.isna(), np.nan, y)

    # 3-level to ordinal: A=0, B=1, C=2 
    if multi_dummy_cols:
        b = df[col_b].astype(float)
        c = df[col_c].astype(float)

        decoded = np.full(len(df), np.nan)
        both_obs = (~b.isna()) & (~c.isna())

        # A: B=0, C=0
        decoded[both_obs & (b == 0) & (c == 0)] = 0
        # B: B=1, C=0
        decoded[both_obs & (b == 1) & (c == 0)] = 1
        # C: B=0, C=1
        decoded[both_obs & (b == 0) & (c == 1)] = 2
        
        decoded[(~b.isna()) & (b == 1) & (c.isna())] = 1
        decoded[(~c.isna()) & (c == 1) & (b.isna())] = 2
        
        # ambiguous cases (b==0, c missing) or (c==0, b missing) stay NaN

        df["categorical_4"] = decoded

    # drop dummy cols
    df.drop(columns=meta["all_dummy_cols"], inplace=True, errors="ignore")
    return df, meta


def ord_to_onehot_df(df_ord: pd.DataFrame, meta: dict):
    """
    Convert ordinal categorical columns back into original one-hot dummy columns.
    This ensures:
    - binary dummies are exactly 0/1 (remain NaN if NaN is input)
    - for categorical_4 at most one of (Level_B, Level_C) is 1 (or both 0 implying Level_A)
    """
    df = df_ord.copy()

    # binary ordinal back to categorical_i_Yes dummy
    for i in range(4):
        ord_col = f"categorical_{i}"
        yes_col = f"categorical_{i}_Yes"
        if ord_col in df.columns:
            v = df[ord_col].to_numpy(dtype=float)
            v = np.where(np.isnan(v), np.nan, np.clip(np.rint(v), 0, 1))

            out = np.full_like(v, np.nan, dtype=float)  
            ok = ~np.isnan(v)
            out[ok] = v[ok].astype(int)

            df[yes_col] = out
            df.drop(columns=[ord_col], inplace=True)

    # 3-level ordinal back to (Level_B, Level_C) dummines
    ord_col = "categorical_4"
    col_b = "categorical_4_Level_B"
    col_c = "categorical_4_Level_C"
    if ord_col in df.columns:
        v = df[ord_col].to_numpy(dtype=float)
        v = np.where(np.isnan(v), np.nan, np.clip(np.rint(v), 0, 2))

        df[col_b] = np.where(np.isnan(v), np.nan, (v == 1).astype(int))
        df[col_c] = np.where(np.isnan(v), np.nan, (v == 2).astype(int))
        df.drop(columns=[ord_col], inplace=True)

    # restore original column order (add any missing columns as NaN)
    for col in meta["original_columns"]:
        if col not in df.columns:
            df[col] = np.nan
    df = df[meta["original_columns"]]
    return df

# Custom KNN Imputer using Gower with Mean/Mode
class KNN_Imputer:
    """
    KNN imputer using Gower metric:

      s_ij = sum_k delta_ijk * s_ijk / sum_k delta_ijk

    delta_ijk = 1 if both values observed for variable k, else 0.

    Variable-wise scores:
      - Quantitative: s_ijk = 1 - |x_i - x_j| / R_k  (range from training)
      - Qualitative:  s_ijk = 1 if equal else 0

    For your synthetic data:
      - input is one-hot
      - convert to ordinal for categoricals (using onehot_to_ord_df)
      - compute similarities + KNN impute in ordinal/numeric space
      - convert back to one-hot (using ord_to_onehot_df)
    """

    def __init__(self, n_neighbors=5):
        self.n_neighbors = int(n_neighbors)

    @staticmethod
    def _mode_ignore_nan(x):
        x = np.asarray(x, dtype=float)
        x = x[~np.isnan(x)]
        if x.size == 0:
            return np.nan
        vals, cnt = np.unique(x, return_counts=True)
        return float(vals[np.argmax(cnt)])

    def _align(self, X_ord: pd.DataFrame) -> pd.DataFrame:
        return X_ord.reindex(columns=self._cols_ord)

    def fit(self, X_train_onehot: pd.DataFrame):
        X_train_ord, meta = onehot_to_ord_df(X_train_onehot)
        self._meta = meta
        self._cols_ord = X_train_ord.columns.tolist()

        # identify categorical ordinal cols
        self._cat_cols = [c for c in self._cols_ord if c.startswith("categorical_")]
        self._num_cols = [c for c in self._cols_ord if c not in self._cat_cols]

        self._train_ord = X_train_ord.copy()
        self._T = self._train_ord.to_numpy(dtype=float)

        # indices for speed
        self._col_index = {c: i for i, c in enumerate(self._cols_ord)}
        self._num_idx = np.array([self._col_index[c] for c in self._num_cols], dtype=int)
        self._cat_idx = np.array([self._col_index[c] for c in self._cat_cols], dtype=int)
        self._cat_set = set(self._cat_idx.tolist())

        # compute numeric ranges R_k from training
        if self._num_idx.size > 0:
            ranges = []
            for c in self._num_cols:
                col = self._train_ord[c].to_numpy(dtype=float)
                # nan-safe min/max
                if np.all(np.isnan(col)):
                    mn, mx = 0.0, 0.0
                else:
                    mn = float(np.nanmin(col))
                    mx = float(np.nanmax(col))
                ranges.append(mx - mn)
            self._R = np.array(ranges, dtype=float)
            self._R_safe = np.where(self._R == 0, 1.0, self._R)
        else:
            self._R = np.array([], dtype=float)
            self._R_safe = np.array([], dtype=float)

        self._fallback = {}

        for c in self._cols_ord:
            col = self._train_ord[c].to_numpy(dtype=float)

            if c in self._cat_cols:
                # categorical → mode
                fb = self._mode_ignore_nan(col)
            else:
                # numeric → mean
                fb = np.nanmean(col) if not np.all(np.isnan(col)) else np.nan

            self._fallback[c] = fb

        return self

    def _similarities_to_train(self, x: np.ndarray) -> np.ndarray:
        """
        Compute similarities s(x, T_r) for all training rows r using Gower (1971).
        """
        T = self._T
        n_train = T.shape[0]

        num_sum = np.zeros(n_train, dtype=float)
        w_sum = np.zeros(n_train, dtype=float)

        # numeric part
        if self._num_idx.size > 0:
            x_num = x[self._num_idx]
            T_num = T[:, self._num_idx]

            valid = (~np.isnan(T_num)) & (~np.isnan(x_num[None, :]))  # delta_ijk
            diff = np.abs(T_num - x_num[None, :])
            s = 1.0 - (diff / self._R_safe[None, :])

            # if any R_k==0 -> similarity 1 if equal else 0 (on valid cells)
            if np.any(self._R == 0):
                zeroR = (self._R == 0)
                eq = (T_num[:, zeroR] == x_num[zeroR][None, :])
                s[:, zeroR] = np.where(eq, 1.0, 0.0)

            s = np.where(valid, s, 0.0)
            num_sum += s.sum(axis=1)
            w_sum += valid.sum(axis=1)

        # categorical part
        if self._cat_idx.size > 0:
            x_cat = x[self._cat_idx]
            T_cat = T[:, self._cat_idx]

            valid = (~np.isnan(T_cat)) & (~np.isnan(x_cat[None, :]))
            s = (T_cat == x_cat[None, :]).astype(float)
            s = np.where(valid, s, 0.0)

            num_sum += s.sum(axis=1)
            w_sum += valid.sum(axis=1)

        # final similarity (undefined when w_sum==0 -> 0)
        sim = np.where(w_sum > 0, num_sum / w_sum, 0.0)
        return sim

    def _impute_ord(self, X_ord: pd.DataFrame) -> pd.DataFrame:
        X_ord = self._align(X_ord)
        X = X_ord.to_numpy(dtype=float)
        T = self._T

        X_imp = X.copy()

        for i in range(X_imp.shape[0]):
            if not np.isnan(X_imp[i]).any():
                continue

            sim = self._similarities_to_train(X_imp[i])
            order = np.argsort(-sim)  # descending similarity

            for j in range(X_imp.shape[1]):
                if not np.isnan(X_imp[i, j]):
                    continue

                neigh = []
                for r in order:
                    if np.isnan(T[r, j]):
                        continue
                    neigh.append(r)
                    if len(neigh) >= self.n_neighbors:
                        break

                colname = self._cols_ord[j]

                # If no donors among kNN → training mean/mode fallback
                if len(neigh) == 0:
                    fb = self._fallback.get(colname, np.nan)
                    if not np.isnan(fb):
                        X_imp[i, j] = fb
                    continue

                vals = T[neigh, j]

                if j in self._cat_set:
                    m = self._mode_ignore_nan(vals)
                    if np.isnan(m):
                        fb = self._fallback.get(colname, np.nan)
                        if not np.isnan(fb):
                            X_imp[i, j] = fb
                    else:
                        X_imp[i, j] = m
                else:
                    mu = float(np.nanmean(vals))
                    if np.isnan(mu):
                        fb = self._fallback.get(colname, np.nan)
                        if not np.isnan(fb):
                            X_imp[i, j] = fb
                    else:
                        X_imp[i, j] = mu

        return pd.DataFrame(X_imp, columns=self._cols_ord, index=X_ord.index)

    def fit_transform(self, X_train_onehot: pd.DataFrame) -> pd.DataFrame:
        self.fit(X_train_onehot)
        X_train_ord, _ = onehot_to_ord_df(X_train_onehot)
        X_train_imp_ord = self._impute_ord(X_train_ord)
        X_train_imp_onehot = ord_to_onehot_df(X_train_imp_ord, self._meta)
        return X_train_imp_onehot[X_train_onehot.columns]

    def transform(self, X_test_onehot: pd.DataFrame) -> pd.DataFrame:
        X_test_ord, _ = onehot_to_ord_df(X_test_onehot)
        X_test_imp_ord = self._impute_ord(X_test_ord)
        X_test_imp_onehot = ord_to_onehot_df(X_test_imp_ord, self._meta)
        return X_test_imp_onehot[X_test_onehot.columns]

def rmse(true, pred):
    return np.sqrt(mean_squared_error(true, pred))

def mae(true, pred):
    return mean_absolute_error(true, pred)


def find_k(k_list, rmse_list, threshold=0.01, min_k=25):
    """
    Pick k (neighbors) value using where improvement starts to flatten out.
    """
    k_list = list(k_list)
    rmse_list = list(rmse_list)

    for i in range(1, len(k_list)):
        prev_k = k_list[i - 1]
        if prev_k < min_k:
            continue

        prev_rmse = rmse_list[i - 1]
        curr_rmse = rmse_list[i]

        if prev_rmse == 0:
            return prev_k

        improvement = (prev_rmse - curr_rmse) / prev_rmse
        if improvement < threshold:
            return prev_k

    valid = [(k, r) for k, r in zip(k_list, rmse_list) if k >= min_k]
    if valid:
        return min(valid, key=lambda x: x[1])[0]

    return k_list[int(np.argmin(rmse_list))]

# Tune KNN_Imputer for one dataset
def tune_knn_iteration(train_missing, train_complete, features,
                       k_candidates, label, iteration,
                       out_dir, threshold=0.01, min_k=25, random_state=57):

    plots_dir = os.path.join(out_dir, "plots")
    os.makedirs(plots_dir, exist_ok=True)

    train_idx, val_idx = train_test_split(
        train_missing.index,
        test_size=0.20,
        random_state=random_state,
        stratify=train_missing["target"]
    )

    train_sub = train_missing.loc[train_idx, features].copy()
    val_sub = train_missing.loc[val_idx, features].copy()

    val_truth = train_complete.loc[val_idx, features].copy()

    missing_mask = val_sub.isna().to_numpy()
    n_missing = int(missing_mask.sum())

    print(f"\nIteration {iteration} | {label}")
    print(f"Validation rows: {len(val_idx)}")
    print(f"Missing cells evaluated: {n_missing}")

    if n_missing == 0:
        default_k = min_k if min_k in k_candidates else k_candidates[0]
        results_df = pd.DataFrame({
            "iteration": iteration,
            "missing_rate": label,
            "k": k_candidates,
            "rmse": [np.nan] * len(k_candidates),
            "mae":  [np.nan] * len(k_candidates),
        })
        return results_df, default_k

    rows = []

    for k in k_candidates:
        imputer = KNN_Imputer(n_neighbors=k)

        # fit on sub-train only, transform validation
        imputer.fit(train_sub)
        val_imputed = imputer.transform(val_sub)

        true = val_truth.to_numpy()[missing_mask]
        pred = val_imputed.to_numpy()[missing_mask]

        r = rmse(true, pred)
        m = mae(true, pred)

        rows.append({
            "iteration": iteration,
            "missing_rate": label,
            "k": k,
            "rmse": float(r),
            "mae": float(m),
        })

        print(f"k={k:>3d}: RMSE={r:.4f}, MAE={m:.4f}")

    results_df = pd.DataFrame(rows)

    best_k = find_k(
        k_list=results_df["k"],
        rmse_list=results_df["rmse"],
        threshold=threshold,
        min_k=min_k
    )

    print(f"Chosen k ({label}) = {best_k}")

    plt.figure()
    plt.plot(results_df["k"], results_df["rmse"], marker="o")
    plt.axvline(best_k, linestyle="--", color="r")
    plt.title(f"Iteration {iteration}: Validation RMSE vs k - {label} missingness")
    plt.xlabel("k (n_neighbors)")
    plt.ylabel("Validation RMSE (missing cells only)")
    plt.tight_layout()

    plot_path = os.path.join(plots_dir, f"iter_{iteration}_{label}_rmse.png")
    plt.savefig(plot_path)
    plt.close()

    return results_df, best_k


# Run tuning for all iterations
def run_knn_tuning(base_dir="data", out_dir="results/knn_tuning",
                   k_candidates=(5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70),
                   threshold=0.01, min_k=25, random_state=57):

    os.makedirs(out_dir, exist_ok=True)

    all_results = []
    best_k_rows = []

    for iteration in range(1, 101):
        print(f"\nTuning Iteration {iteration}")

        complete_dir = os.path.join(base_dir, "complete", f"iteration_{iteration}")
        amputed_dir = os.path.join(base_dir, "amputed", f"iteration_{iteration}")

        train_complete = pd.read_csv(os.path.join(complete_dir, "train_complete.csv"))
        train_low = pd.read_csv(os.path.join(amputed_dir, "train_low.csv"))
        train_high = pd.read_csv(os.path.join(amputed_dir, "train_high.csv"))

        features = [c for c in train_low.columns if c != "target"]

        low_df, best_low = tune_knn_iteration(
            train_missing=train_low,
            train_complete=train_complete,
            features=features,
            k_candidates=list(k_candidates),
            label="10%",
            iteration=iteration,
            out_dir=out_dir,
            threshold=threshold,
            min_k=min_k,
            random_state=random_state
        )

        high_df, best_high = tune_knn_iteration(
            train_missing=train_high,
            train_complete=train_complete,
            features=features,
            k_candidates=list(k_candidates),
            label="25%",
            iteration=iteration,
            out_dir=out_dir,
            threshold=threshold,
            min_k=min_k,
            random_state=random_state
        )

        all_results.append(low_df)
        all_results.append(high_df)

        best_k_rows.append({"iteration": iteration, "missing_rate": "10%", "best_k": int(best_low)})
        best_k_rows.append({"iteration": iteration, "missing_rate": "25%", "best_k": int(best_high)})

        results_df = pd.concat(all_results, ignore_index=True)
        best_df = pd.DataFrame(best_k_rows)

        results_df.to_csv(os.path.join(out_dir, "knn_tuning_all_iterations.csv"), index=False)
        best_df.to_csv(os.path.join(out_dir, "knn_tuning_summary.csv"), index=False)

    return pd.concat(all_results, ignore_index=True), pd.DataFrame(best_k_rows)


if __name__ == "__main__":
    all_df, best_df = run_knn_tuning()
    print("\nFinished KNN tuning.")
    print("Saved: results/knn_tuning/knn_tuning_all_iterations.csv")
    print("Saved: results/knn_tuning/knn_tuning_summary.csv")
    print("Saved plots: results/knn_tuning/plots/iter_*_10%_rmse.png and iter_*_25%_rmse.png")


Tuning Iteration 1

Iteration 1 | 10%
Validation rows: 140
Missing cells evaluated: 455
k=  5: RMSE=0.9633, MAE=0.7457
k= 10: RMSE=0.9238, MAE=0.7104
k= 15: RMSE=0.8999, MAE=0.6899
k= 20: RMSE=0.8947, MAE=0.6832
k= 25: RMSE=0.8978, MAE=0.6851
k= 30: RMSE=0.9001, MAE=0.6853
k= 35: RMSE=0.8948, MAE=0.6770
k= 40: RMSE=0.8991, MAE=0.6866
k= 45: RMSE=0.8950, MAE=0.6764
k= 50: RMSE=0.8926, MAE=0.6745
k= 55: RMSE=0.8905, MAE=0.6688
k= 60: RMSE=0.8906, MAE=0.6735
k= 65: RMSE=0.8940, MAE=0.6807
k= 70: RMSE=0.8936, MAE=0.6797
Chosen k (10%) = 25

Iteration 1 | 25%
Validation rows: 140
Missing cells evaluated: 910
k=  5: RMSE=0.9532, MAE=0.7182
k= 10: RMSE=0.9193, MAE=0.6974
k= 15: RMSE=0.9132, MAE=0.6882
k= 20: RMSE=0.9052, MAE=0.6741
k= 25: RMSE=0.9044, MAE=0.6747
k= 30: RMSE=0.9048, MAE=0.6757
k= 35: RMSE=0.9079, MAE=0.6792
k= 40: RMSE=0.9035, MAE=0.6716
k= 45: RMSE=0.9004, MAE=0.6688
k= 50: RMSE=0.9036, MAE=0.6710
k= 55: RMSE=0.9022, MAE=0.6686
k= 60: RMSE=0.9059, MAE=0.6753
k= 65: RMSE=0.90

In [4]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

try:
    import sklearn.neighbors._base
    sys.modules["sklearn.neighbors.base"] = sklearn.neighbors._base
except Exception:
    pass

from missingpy import MissForest

warnings.filterwarnings("ignore", category=FutureWarning)

# Metrics (missing cells only) 
def RMSE(original, imputed, mask):
    if mask.sum() == 0:
        return np.nan
    return np.sqrt(mean_squared_error(original[mask], imputed[mask]))

def MAE(original, imputed, mask):
    if mask.sum() == 0:
        return np.nan
    return mean_absolute_error(original[mask], imputed[mask])

def R2(original, imputed, mask):
    # r2_score needs >= 2 samples
    if mask.sum() < 2:
        return np.nan
    return r2_score(original[mask], imputed[mask])


# Tuned k
def load_k_table(tuned_k_csv):
    if not os.path.exists(tuned_k_csv):
        raise FileNotFoundError(f"Missing tuned k file: {tuned_k_csv}")

    df = pd.read_csv(tuned_k_csv)
    required = {"iteration", "missing_rate", "best_k"}
    if not required.issubset(df.columns):
        raise ValueError(f"{tuned_k_csv} must contain columns: {sorted(required)}")

    tuned_k = {}
    for _, row in df.iterrows():
        it = int(row["iteration"])
        mr = str(row["missing_rate"]).strip().lower()
        tuned_k[(it, mr)] = int(row["best_k"])
    return tuned_k

# Imputation function for a single dataset
def impute(iteration, missing_rate,
           X_train_missing, X_test_missing,
           X_train_complete, X_test_complete,
           k_tuned, random_state=57):

    results = []

    imputed_dir = f"data/imputed/iteration_{iteration}"
    os.makedirs(imputed_dir, exist_ok=True)

    mask_train = X_train_missing.isna().to_numpy()
    mask_test = X_test_missing.isna().to_numpy()

    methods = ["Simple", "KNN", "MissForest"]

    for method in methods:

        if method == "KNN":
            print(f" - {method} using tuned k={k_tuned}")
        else:
            print(f" - {method}")

        if method == "Simple":
            # onehot -> ordinal
            X_train_ord, meta = onehot_to_ord_df(X_train_missing)
            X_test_ord, _ = onehot_to_ord_df(X_test_missing)

            cat_cols = [c for c in X_train_ord.columns if c.startswith("categorical_")]
            num_cols = [c for c in X_train_ord.columns if c not in cat_cols]

            num_imputer = SimpleImputer(strategy="mean")
            cat_imputer = SimpleImputer(strategy="most_frequent")

            X_train_imp_ord = X_train_ord.copy()
            if num_cols:
                X_train_imp_ord[num_cols] = num_imputer.fit_transform(X_train_imp_ord[num_cols])
            if cat_cols:
                X_train_imp_ord[cat_cols] = cat_imputer.fit_transform(X_train_imp_ord[cat_cols])

            X_test_imp_ord = X_test_ord.copy()
            if num_cols:
                X_test_imp_ord[num_cols] = num_imputer.transform(X_test_imp_ord[num_cols])
            if cat_cols:
                X_test_imp_ord[cat_cols] = cat_imputer.transform(X_test_imp_ord[cat_cols])

            # ordinal -> onehot
            X_train_imp_df = ord_to_onehot_df(X_train_imp_ord, meta)[X_train_missing.columns]
            X_test_imp_df = ord_to_onehot_df(X_test_imp_ord, meta)[X_test_missing.columns]

        elif method == "KNN":
            knn = KNN_Imputer(n_neighbors=k_tuned)

            print("   Fit on train, transform train")
            knn.fit(X_train_missing)
            X_train_imp_df = knn.transform(X_train_missing)

            print("   Transform test")
            X_test_imp_df = knn.transform(X_test_missing)

            # safety: enforce same columns/order
            X_train_imp_df = X_train_imp_df[X_train_missing.columns]
            X_test_imp_df = X_test_imp_df[X_test_missing.columns]

        elif method == "MissForest":
            X_train_mf, meta = onehot_to_ord_df(X_train_missing)
            X_test_mf, _ = onehot_to_ord_df(X_test_missing)

            cat_cols = [c for c in X_train_mf.columns if c.startswith("categorical_")]
            cat_vars = [X_train_mf.columns.get_loc(c) for c in cat_cols] if cat_cols else None

            mf = MissForest(random_state=random_state)

            print("   Fit + transform train")
            X_train_imp_mf = mf.fit_transform(X_train_mf.to_numpy(), cat_vars=cat_vars)

            print("   Transform test")
            X_test_imp_mf = mf.transform(X_test_mf.to_numpy())

            X_train_imp_mf_df = pd.DataFrame(X_train_imp_mf, columns=X_train_mf.columns, index=X_train_mf.index)
            X_test_imp_mf_df = pd.DataFrame(X_test_imp_mf, columns=X_test_mf.columns, index=X_test_mf.index)

            X_train_imp_df = ord_to_onehot_df(X_train_imp_mf_df, meta)[X_train_missing.columns]
            X_test_imp_df = ord_to_onehot_df(X_test_imp_mf_df, meta)[X_test_missing.columns]

        else:
            raise ValueError(f"Unknown method: {method}")

        # save imputed datasets
        X_train_imp_df.to_csv(f"{imputed_dir}/train_{missing_rate}_{method.lower()}.csv", index=False)
        X_test_imp_df.to_csv(f"{imputed_dir}/test_{missing_rate}_{method.lower()}.csv", index=False)

        # metrics on missing cells only
        X_train_imp = X_train_imp_df.to_numpy()
        X_test_imp = X_test_imp_df.to_numpy()

        for split, X_true, X_imp, mask in [
            ("train", X_train_complete.to_numpy(), X_train_imp, mask_train),
            ("test",  X_test_complete.to_numpy(),  X_test_imp,  mask_test),
        ]:
            results.append({
                "iteration": iteration,
                "missing_rate": missing_rate,
                "method": method,
                "split": split,
                "rmse": RMSE(X_true, X_imp, mask),
                "mae": MAE(X_true, X_imp, mask),
                "r2": R2(X_true, X_imp, mask),
            })

    return results

# Run all iterations
def run_all(
    base_dir="data",
    k_tuned_csv="results/knn_tuning/knn_tuning_summary.csv",
    output_path="results/imputation_results.csv",
    random_state=57):

    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    tuned_k = load_k_table(k_tuned_csv)
    all_results = []

    for iteration in range(1, 101):

        complete_dir = os.path.join(base_dir, "complete", f"iteration_{iteration}")
        amputed_dir = os.path.join(base_dir, "amputed", f"iteration_{iteration}")

        train_complete = pd.read_csv(os.path.join(complete_dir, "train_complete.csv"))
        test_complete = pd.read_csv(os.path.join(complete_dir, "test_complete.csv"))

        X_train_complete = train_complete.drop(columns="target")
        X_test_complete = test_complete.drop(columns="target")

        train_low = pd.read_csv(os.path.join(amputed_dir, "train_low.csv")).drop(columns="target")
        test_low = pd.read_csv(os.path.join(amputed_dir, "test_low.csv")).drop(columns="target")
        train_high = pd.read_csv(os.path.join(amputed_dir, "train_high.csv")).drop(columns="target")
        test_high = pd.read_csv(os.path.join(amputed_dir, "test_high.csv")).drop(columns="target")

        k_low = tuned_k[(iteration, "10%")]
        k_high = tuned_k[(iteration, "25%")]

        print(f"\nIteration {iteration}")

        print("Imputing missing rate 10% (low)")
        all_results.extend(impute(
            iteration, "10%",
            train_low, test_low,
            X_train_complete, X_test_complete,
            k_low, random_state
        ))

        print("Imputing missing rate 25% (high)")
        all_results.extend(impute(
            iteration, "25%",
            train_high, test_high,
            X_train_complete, X_test_complete,
            k_high, random_state
        ))

        df_out = pd.DataFrame(all_results)
        df_out = df_out[["iteration", "split", "missing_rate", "method", "rmse", "mae", "r2"]]
        df_out.to_csv(output_path, index=False)

    print("\nSaved:", output_path)
    return pd.DataFrame(all_results)


if __name__ == "__main__":
    run_all()
    print("\nSuccessfully completed imputation for all 100 iterations.")


Iteration 89
Imputing missing rate 10% (low)
 - Simple
 - KNN using tuned k=25
   Fit on train, transform train
   Transform test
 - MissForest
   Fit + transform train
Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 3
Iteration: 4
   Transform test
Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 3
Iteration: 4
Iteration: 5
Imputing missing rate 25% (high)
 - Simple
 - KNN using tuned k=25
   Fit on train, transform train
   Transform test
 - MissForest
   Fit + transform train
Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 3
Iteration: 4
Iteration: 5
Iteration: 6
Iteration: 7
Iteration: 8
Iteration: 9
   Transform test
Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 3
Iteration: 4
Iteration: 5

Iteration 90
Imputing missing rate 10% (low)
 - Simple
 - KNN using tuned k=25
   Fit on train, transform train
   Transform test
 - MissForest
   Fit + transform train
Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 3
   Transform test
Iteration: 0
Iteration: 1
Iteration: 2
Iterati

In [5]:
import os
import pandas as pd
import numpy as np
from scipy.stats import ks_2samp

base_dir = "data"
output_per_feature = "results/ks_per_feature.csv"
output_summary = "results/ks_summary.csv"

def KS_per_feature(original_df, imputed_df, mask_df):
    """
    Calculate KS statistic per feature using positions that were amputed.
    Returns dict: {feature: ks_stat (or np.nan if not enough missing)}
    """
    ks_dict = {}

    for col in original_df.columns:
        mask = mask_df[col].isna().to_numpy()

        # need at least 2 missing values to run KS sensibly
        if mask.sum() >= 2:
            stat, pvalue = ks_2samp(original_df.loc[mask, col], imputed_df.loc[mask, col])
            ks_dict[col] = stat
        else:
            ks_dict[col] = np.nan

    return ks_dict

def run_ks():
    all_rows = []

    for iteration in range(1, 101):
        print(f"\nIteration {iteration}")

        complete_dir = f"{base_dir}/complete/iteration_{iteration}"
        amputed_dir = f"{base_dir}/amputed/iteration_{iteration}"
        imputed_dir = f"{base_dir}/imputed/iteration_{iteration}"

        # skip if iteration folders don't exist
        if not (os.path.isdir(complete_dir) and os.path.isdir(amputed_dir) and os.path.isdir(imputed_dir)):
            print("Skipping (missing folders for iteration)")
            continue

        # load complete
        train_complete_path = f"{complete_dir}/train_complete.csv"
        test_complete_path = f"{complete_dir}/test_complete.csv"

        if not (os.path.exists(train_complete_path) and os.path.exists(test_complete_path)):
            print("Skipping (missing complete files)")
            continue

        train_complete = pd.read_csv(train_complete_path)
        test_complete = pd.read_csv(test_complete_path)

        X_train_complete = train_complete.drop(columns="target")
        X_test_complete = test_complete.drop(columns="target")

        feature_cols = X_train_complete.columns.tolist()

        configs = [
            ("10%", "low"),
            ("25%", "high")
        ]

        for missing_rate, suffix in configs:

            train_miss_path = f"{amputed_dir}/train_{suffix}.csv"
            test_miss_path = f"{amputed_dir}/test_{suffix}.csv"

            if not (os.path.exists(train_miss_path) and os.path.exists(test_miss_path)):
                print(f"Skipping {missing_rate} (missing amputed files)")
                continue

            X_train_miss = pd.read_csv(train_miss_path).drop(columns="target")
            X_test_miss = pd.read_csv(test_miss_path).drop(columns="target")

            X_train_miss = X_train_miss[feature_cols]
            X_test_miss = X_test_miss[feature_cols]

            for method in ["simple", "knn", "missforest"]:

                train_imp_path = f"{imputed_dir}/train_{missing_rate}_{method}.csv"
                test_imp_path = f"{imputed_dir}/test_{missing_rate}_{method}.csv"

                if not (os.path.exists(train_imp_path) and os.path.exists(test_imp_path)):
                    print(f"Skipping {missing_rate} {method} (missing imputed files)")
                    continue

                # load imputed files
                X_train_imp = pd.read_csv(train_imp_path)[feature_cols]
                X_test_imp = pd.read_csv(test_imp_path)[feature_cols]

                # KS per feature (using amputed NaNs as mask)
                ks_train = KS_per_feature(X_train_complete, X_train_imp, X_train_miss)
                ks_test = KS_per_feature(X_test_complete,  X_test_imp,  X_test_miss)

                for feat, ks_val in ks_train.items():
                    all_rows.append({
                        "iteration": iteration,
                        "split": "train",
                        "missing_rate": missing_rate,
                        "method": method,
                        "feature": feat,
                        "ks": ks_val
                    })

                for feat, ks_val in ks_test.items():
                    all_rows.append({
                        "iteration": iteration,
                        "split": "test",
                        "missing_rate": missing_rate,
                        "method": method,        
                        "feature": feat,
                        "ks": ks_val
                    })

    ks_df = pd.DataFrame(all_rows)
    ks_df.to_csv(output_per_feature, index=False)

    # summary per (iteration, split, missingness, method)
    summary_df = (
        ks_df
        .groupby(["iteration", "split", "missing_rate", "method"], as_index=False)
        .agg(
            ks_mean=("ks", "mean"),
            ks_median=("ks", "median"),
            ks_max=("ks", "max"),
            n_features=("ks", lambda s: int(np.isfinite(s).sum()))
        )
    )
    summary_df.to_csv(output_summary, index=False)

    print(f"\nSaved per-feature KS to: {output_per_feature}")
    print(f"Saved KS summary to: {output_summary}")

    return ks_df, summary_df

if __name__ == "__main__":
    ks_df, summary_df = run_ks()

[autoreload of sklearn.utils.class_weight failed: Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/opt/conda/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 500, in superreload
    update_generic(old_obj, new_obj)
  File "/opt/conda/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 397, in update_generic
    update(a, b)
  File "/opt/conda/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 309, in update_function
    setattr(old, name, getattr(new, name))
ValueError: compute_class_weight() requires a code object with 0 free vars, not 3
]
[autoreload of sklearn.utils.fixes failed: Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/opt/conda/lib/python3.


Iteration 1


/opt/conda/lib/python3.11/site-packages/scipy/stats/_axis_nan_policy.py:531: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)
/opt/conda/lib/python3.11/site-packages/scipy/stats/_axis_nan_policy.py:531: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)
/opt/conda/lib/python3.11/site-packages/scipy/stats/_axis_nan_policy.py:531: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)



Iteration 2

Iteration 3


/opt/conda/lib/python3.11/site-packages/scipy/stats/_axis_nan_policy.py:531: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)



Iteration 4

Iteration 5


KeyboardInterrupt: 

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp

def featurewise_ks_for_one_dataset(
    X_complete: pd.DataFrame,
    X_missing: pd.DataFrame,
    X_imputed: pd.DataFrame,
    iteration: int,
    split: str,
    missing_rate: str,
    method: str,
    alpha: float = 0.05,
    min_n: int = 5
) -> pd.DataFrame:
    """
    Feature-wise two-sample KS test on *missing cells only*.
    Compares: true values (from complete data) vs imputed values (from imputed data),
    restricted to entries that were missing in X_missing.

    Returns one row per feature with KS statistic and p-value.
    """
    rows = []
    for col in X_complete.columns:
        miss_mask = X_missing[col].isna().to_numpy()
        n_miss = int(miss_mask.sum())

        if n_miss < min_n:
            rows.append({
                "iteration": iteration,
                "split": split,
                "missing_rate": missing_rate,
                "method": method,
                "feature": col,
                "n_missing": n_miss,
                "ks_stat": np.nan,
                "p_value": np.nan,
                "reject_05": np.nan,
            })
            continue

        true_vals = X_complete[col].to_numpy(dtype=float)[miss_mask]
        imp_vals  = X_imputed[col].to_numpy(dtype=float)[miss_mask]

        # Drop NaNs (should be none in imp_vals, but safe)
        true_vals = true_vals[~np.isnan(true_vals)]
        imp_vals  = imp_vals[~np.isnan(imp_vals)]

        if len(true_vals) < min_n or len(imp_vals) < min_n:
            ks_stat, p_val = np.nan, np.nan
            reject = np.nan
        else:
            res = ks_2samp(true_vals, imp_vals, alternative="two-sided", mode="auto")
            ks_stat, p_val = float(res.statistic), float(res.pvalue)
            reject = bool(p_val < alpha)

        rows.append({
            "iteration": iteration,
            "split": split,
            "missing_rate": missing_rate,
            "method": method,
            "feature": col,
            "n_missing": n_miss,
            "ks_stat": ks_stat,
            "p_value": p_val,
            "reject_05": reject,
        })

    return pd.DataFrame(rows)


def compute_featurewise_ks_all(
    base_dir: str = "data",
    output_csv: str = "results/featurewise_ks_results.csv",
    alpha: float = 0.05,
    min_n: int = 5,
    iterations: range = range(1, 101),
    missing_rates = ("10%", "25%"),
    methods = ("simple", "knn", "missforest")
):
    os.makedirs(os.path.dirname(output_csv), exist_ok=True)

    all_rows = []

    for it in iterations:
        complete_dir = os.path.join(base_dir, "complete", f"iteration_{it}")
        amputed_dir  = os.path.join(base_dir, "amputed",  f"iteration_{it}")
        imputed_dir  = os.path.join(base_dir, "imputed",  f"iteration_{it}")

        # complete
        train_complete = pd.read_csv(os.path.join(complete_dir, "train_complete.csv"))
        test_complete  = pd.read_csv(os.path.join(complete_dir, "test_complete.csv"))

        X_train_complete = train_complete.drop(columns="target")
        X_test_complete  = test_complete.drop(columns="target")

        # missing datasets (amputed)
        train_low  = pd.read_csv(os.path.join(amputed_dir, "train_low.csv")).drop(columns="target")
        test_low   = pd.read_csv(os.path.join(amputed_dir, "test_low.csv")).drop(columns="target")
        train_high = pd.read_csv(os.path.join(amputed_dir, "train_high.csv")).drop(columns="target")
        test_high  = pd.read_csv(os.path.join(amputed_dir, "test_high.csv")).drop(columns="target")

        missing_map = {
            "10%": (train_low, test_low),
            "25%": (train_high, test_high),
        }

        for mr in missing_rates:
            X_train_missing, X_test_missing = missing_map[mr]

            for method in methods:
                # your saved filenames: train_{missing_rate}_{method}.csv
                train_imp_path = os.path.join(imputed_dir, f"train_{mr}_{method}.csv")
                test_imp_path  = os.path.join(imputed_dir, f"test_{mr}_{method}.csv")

                if not (os.path.exists(train_imp_path) and os.path.exists(test_imp_path)):
                    raise FileNotFoundError(f"Missing imputed files for it={it}, mr={mr}, method={method}")

                X_train_imp = pd.read_csv(train_imp_path)
                X_test_imp  = pd.read_csv(test_imp_path)

                # Ensure same column order
                X_train_imp = X_train_imp[X_train_complete.columns]
                X_test_imp  = X_test_imp[X_test_complete.columns]

                # Train KS
                df_train = featurewise_ks_for_one_dataset(
                    X_complete=X_train_complete,
                    X_missing=X_train_missing,
                    X_imputed=X_train_imp,
                    iteration=it,
                    split="train",
                    missing_rate=mr,
                    method=method.capitalize() if method != "knn" else "KNN",
                    alpha=alpha,
                    min_n=min_n
                )

                # Test KS
                df_test = featurewise_ks_for_one_dataset(
                    X_complete=X_test_complete,
                    X_missing=X_test_missing,
                    X_imputed=X_test_imp,
                    iteration=it,
                    split="test",
                    missing_rate=mr,
                    method=method.capitalize() if method != "knn" else "KNN",
                    alpha=alpha,
                    min_n=min_n
                )

                all_rows.append(df_train)
                all_rows.append(df_test)

        # incremental save (safe if interrupted)
        out_df = pd.concat(all_rows, ignore_index=True)
        out_df.to_csv(output_csv, index=False)
        print(f"Iteration {it}: saved {len(out_df):,} rows -> {output_csv}")

    print("\nDone.")
    print("Saved:", output_csv)
    return pd.concat(all_rows, ignore_index=True)


if __name__ == "__main__":
    compute_featurewise_ks_all()
